# Getting started with SpikingGamma

This notebook will help you to get started with SpikingGamma. Besides the spikinggamma package, only PyTorch is a strict requirement, but for this notebook you also need tonic.
It is highly recommended to use a CUDA GPU. Make sure you have installed the correct version of PyTorch for this.

In [1]:
import torch
import torch.nn as nn
import tonic
from torch.utils.data import DataLoader
from tqdm import tqdm
from spikinggamma.neuron import SpikingGammaNeuron, SpikingGammaOutput
from spikinggamma.spike.sigma_delta import SigmaDeltaSpikeFunction
from spikinggamma.dendrite.split import SplitDendriticFunction
from spikinggamma.dendrite.synapse.linear import LinearPerNeuron
from spikinggamma.signal.default import DefaultSignalFunction
from spikinggamma.spike.base import compute_linspace_bucket_decay_rates, SpikeConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


## Loading the data
We use SHD as example, with 250 timesteps. The number of features is scaled down to 140. For performance reasons, caching is used through tonic's `DiskCachedDataset`. This makes the first epoch much slower.

In [2]:
from load_data import get_shd_dataset
train_dataset, test_dataset, input_shape, n_outputs, T = get_shd_dataset(timestep_size=3600)

## Neuron structure

A neuron has three components: spike function, dendritic (synaptic) function, and signal function.  
The `SpikingGammaNeuron` class processes input in three steps:

1. First, the input goes to the spike function, producing spikes and bucket values. This decomposes the input signal into bucket components for downstream processing.
2. Next, the bucket components (**not** the spikes, they are not used for training) are passed to the dendritic function, which applies a synaptic transform (e.g., linear or convolution) to each bucket component independently. The results are then weighted per component and summed such that there is one dendritic output value per neuron.
3. Finally, the dendritic output is processed by the signal function to produce a final neuron signal. No spikes are emitted at this stage.

To get spiking output, an extra `SpikingGammaOutput` layer is used to convert the final neuron signal to spikes/bucket components.


### Tips for building a network
`SpikingGammaNeuron` can be used like any other layer. Just remember that it contains both the synaptic transformation and the spiking activation. It is possible to do more transformations on the neuron signal in between the neuron layers, more about this in a later example notebook. Remember to allow for resetting the network (and to call it), this is very important to do in between samples!

In [3]:
# Hyperparameters
n_buckets = 10
decay_factor = 0.1
spike_config = SpikeConfig(
    spike_threshold=0.3
)
bucket_decay_rates = compute_linspace_bucket_decay_rates(n_buckets, decay_factor)

# A simple fully connected network with 2 hidden layers
class SpikingGammaNet(nn.Module):
    def __init__(self, 
                 input_shape, 
                 n_outputs,
                 n_buckets,
                 bucket_decay_rates,
                 spike_config):
        super().__init__()
        layer1_shape_out = (256,)
        layer2_shape_out = (256,)
        layer3_shape_out = (n_outputs,)

        self.hid1 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer1_shape_out,
                n_buckets=n_buckets,
                synaptic_function=LinearPerNeuron(input_shape[0], layer1_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(layer1_shape_out))
        )

        self.hid2 = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer2_shape_out,
                n_buckets=n_buckets,
                synaptic_function=LinearPerNeuron(layer1_shape_out[0], layer2_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(layer2_shape_out))
        )

        self.out = SpikingGammaNeuron(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config),
            dendritic_fn=SplitDendriticFunction(
                layer3_shape_out,
                n_buckets=n_buckets,
                synaptic_function=LinearPerNeuron(layer2_shape_out[0], layer3_shape_out[0])
            ),
            signal_fn=DefaultSignalFunction(norm_fn=nn.LayerNorm(layer3_shape_out))
        )

        self.spike_out = SpikingGammaOutput(
            spike_fn=SigmaDeltaSpikeFunction(bucket_decay_rates, spike_config)
        )

    def reset(self):
        self.hid1.reset()
        self.hid2.reset()
        self.out.reset()
        self.spike_out.reset()

    def detach(self):
        self.hid1.detach()
        self.hid2.detach()
        self.out.detach()
        self.spike_out.detach()

    def forward(self, s_in):
        s_in = s_in.flatten(start_dim=1)
        y_hid1 = self.hid1(s_in)
        y_hid2 = self.hid2(y_hid1)
        y_out = self.out(y_hid2)
        y_hat_out, s_out = self.spike_out(y_out)
        return y_out, y_hat_out, s_out

## Training the network
Training is fairly straightforward. The usual PyTorch / deep learning mechanisms apply. Remember we do online learning, so the loss is computed per timestep.

In [4]:
train_batch_size = 32
test_batch_size = 128
train_dataloader = DataLoader(train_dataset, batch_size=train_batch_size, shuffle=True, collate_fn=tonic.collation.PadTensors(batch_first=False))
test_dataloader = DataLoader(test_dataset, batch_size=test_batch_size, shuffle=False, collate_fn=tonic.collation.PadTensors(batch_first=False))
model = SpikingGammaNet(input_shape, n_outputs, n_buckets, bucket_decay_rates, spike_config).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

In [5]:
model.train()
n_epochs = 3
for _ in range(n_epochs):
    batch_iter = iter(train_dataloader)
    pbar = tqdm(batch_iter, total=len(train_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        optimizer.zero_grad()
        total_loss = 0
        for t in range(data.shape[0]):
            y, y_hat, s = model(data[t])
            loss = loss_fn(y, labels) * 1/data.shape[0]
            loss.backward()
            total_loss += loss.item()

        optimizer.step()
        pbar.set_description(f"Loss: {total_loss:.4f}")

Loss: 1.3782: 100%|██████████| 255/255 [01:11<00:00,  3.55it/s]


Remember that the first epoch may be slower because of tonic's `DiskCachedDataset` that is enabled by default.

## Testing
For computing the accuracy, we can count the spikes at the output. Three epochs should result in about 70% accuracy with this network.

In [9]:
batch_iter = iter(test_dataloader)
model.eval()
correct = 0
total = 0
with torch.no_grad():
    pbar = tqdm(batch_iter, total=len(test_dataloader))
    for data, labels in pbar:
        data = data.to(device)
        labels = labels.to(device)

        model.reset()
        s_list = [] 
        for t in range(data.shape[0]):
            y, y_hat, s = model(data[t])
            s_list.append(s)
        s = torch.stack(s_list, dim=0)
        predictions = s.sum(dim=0).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
accuracy = correct / total
print(f'Test Accuracy: {accuracy * 100:.2f}%')

100%|██████████| 18/18 [00:03<00:00,  5.77it/s]

Test Accuracy: 71.07%


You can save and load the model through the usual PyTorch module functionality.

In [7]:
# Save
torch.save(model.state_dict(), "spiking_gamma_model.pth")

In [8]:
# Load
model = SpikingGammaNet(input_shape, n_outputs, n_buckets, bucket_decay_rates, spike_config).to(device)
model.load_state_dict(torch.load("spiking_gamma_model.pth"))

<All keys matched successfully>